# Поиск ассоциативных правил (Apriori)

Доработка лабораторной «Поиск частых наборов»: по частым наборам строятся ассоциативные правила вида **антецедент → консеквент** с поддержкой и достоверностью.

Алгоритм Apriori и генерация правил реализованы самостоятельно.

**Параметры программы:**
- набор транзакций;
- порог поддержки `min_support`;
- порог достоверности `min_confidence`;
- упорядочивание правил: по убыванию поддержки (`support_desc`) или лексикографически (`lex`).


## Определения

Поддержка набора $X$:

$$\mathrm{supp}(X) = \frac{|\{t : X \subseteq t\}|}{|T|}$$

Правило $A \rightarrow B$ (непересекающиеся непустые $A$, $B$) строится только если набор $A \cup B$ частый. Достоверность:

$$\mathrm{conf}(A \rightarrow B) = \frac{\mathrm{supp}(A \cup B)}{\mathrm{supp}(A)}$$

Правило оставляем, если $\mathrm{conf}(A \rightarrow B) \ge \mathrm{min\_confidence}$.


In [ ]:
from collections import defaultdict
from itertools import combinations
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11
plt.rcParams["font.family"] = "DejaVu Sans"


## Загрузка данных (`baskets.csv`, кодировка Windows-1251)


In [ ]:
def load_transactions(path):
    transactions = []
    with open(path, encoding="cp1251") as f:
        for line in f:
            items = {item.strip() for item in line.strip().split(",") if item.strip()}
            if items:
                transactions.append(items)
    return transactions


DATA_PATH = "baskets.csv"
transactions = load_transactions(DATA_PATH)
print(f"Транзакций: {len(transactions)}")
print(f"Уникальных товаров: {len({i for t in transactions for i in t})}")
print(f"Средняя длина корзины: {sum(len(t) for t in transactions) / len(transactions):.2f}")


## Apriori (как в лабораторной 1)


In [ ]:
def _apriori_join(prev_level, k):
    candidates = set()
    m = len(prev_level)
    for i in range(m):
        for j in range(i + 1, m):
            union = prev_level[i] | prev_level[j]
            if len(union) == k:
                candidates.add(union)
    return list(candidates)


def _apriori_prune(candidates, prev_frequent, k):
    kept = []
    for cand in candidates:
        if all(frozenset(sub) in prev_frequent for sub in combinations(cand, k - 1)):
            kept.append(cand)
    return kept


def apriori(transactions, min_support, order="support_desc"):
    if order not in {"support_desc", "lex"}:
        raise ValueError('order должен быть "support_desc" или "lex"')

    n_trans = len(transactions)
    if n_trans == 0:
        return []

    min_count = min_support * n_trans
    support = {}

    item_counts = defaultdict(int)
    for basket in transactions:
        for item in basket:
            item_counts[item] += 1

    L_current = []
    for item, cnt in item_counts.items():
        if cnt >= min_count:
            itemset = frozenset([item])
            L_current.append(itemset)
            support[itemset] = cnt / n_trans
    L_current.sort(key=lambda s: tuple(sorted(s)))

    all_frequent = list(L_current)
    k = 2
    while L_current:
        candidates = _apriori_join(L_current, k)
        candidates = _apriori_prune(candidates, set(L_current), k)
        if not candidates:
            break

        cand_index = set(candidates)
        counts = defaultdict(int)
        for basket in transactions:
            if len(basket) < k:
                continue
            for comb in combinations(sorted(basket), k):
                itemset = frozenset(comb)
                if itemset in cand_index:
                    counts[itemset] += 1

        L_next = []
        for cand in candidates:
            cnt = counts[cand]
            if cnt >= min_count:
                L_next.append(cand)
                support[cand] = cnt / n_trans

        L_next.sort(key=lambda s: tuple(sorted(s)))
        all_frequent.extend(L_next)
        L_current = L_next
        k += 1

    result = [(itemset, support[itemset]) for itemset in all_frequent]
    if order == "support_desc":
        result.sort(key=lambda x: (-x[1], tuple(sorted(x[0]))))
    else:
        result.sort(key=lambda x: tuple(sorted(x[0])))
    return result


## Генерация ассоциативных правил

Для каждого частого набора длины $\ge 2$ перебираются непустые собственные поднаборы как антецедент; консеквент — оставшиеся объекты. Правило сохраняется, если достоверность не ниже порога.


In [ ]:
def _fmt_itemset(itemset):
    return ", ".join(sorted(itemset))


def association_rules(frequent_itemsets, min_confidence, order="support_desc"):
    """Правила по списку частых наборов (itemset, support)."""
    if order not in {"support_desc", "lex"}:
        raise ValueError('order должен быть "support_desc" или "lex"')

    support = {itemset: supp for itemset, supp in frequent_itemsets}
    rules = []
    for itemset, supp in frequent_itemsets:
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ante in combinations(items, r):
                antecedent = frozenset(ante)
                consequent = itemset - antecedent
                conf = supp / support[antecedent]
                if conf >= min_confidence:
                    rules.append(
                        {
                            "antecedent": antecedent,
                            "consequent": consequent,
                            "support": supp,
                            "confidence": conf,
                        }
                    )

    if order == "support_desc":
        rules.sort(
            key=lambda r: (
                -r["support"],
                -r["confidence"],
                tuple(sorted(r["antecedent"])),
                tuple(sorted(r["consequent"])),
            )
        )
    else:
        rules.sort(
            key=lambda r: (
                tuple(sorted(r["antecedent"])),
                tuple(sorted(r["consequent"])),
            )
        )
    return rules


def mine_association_rules(transactions, min_support, min_confidence, order="support_desc"):
    """Полный конвейер: частые наборы + правила."""
    itemsets = apriori(transactions, min_support, order="support_desc")
    return association_rules(itemsets, min_confidence, order=order), itemsets


def rules_to_frame(rules):
    rows = []
    for r in rules:
        rows.append(
            {
                "правило": f"{_fmt_itemset(r['antecedent'])} → {_fmt_itemset(r['consequent'])}",
                "антецедент": _fmt_itemset(r["antecedent"]),
                "консеквент": _fmt_itemset(r["consequent"]),
                "длина A∪B": len(r["antecedent"]) + len(r["consequent"]),
                "поддержка": r["support"],
                "достоверность": r["confidence"],
            }
        )
    return pd.DataFrame(rows)


def show_df(df, formats=None):
    """Показать таблицу без pandas Styler."""
    out = df.copy()
    if formats:
        for col, spec in formats.items():
            if col in out.columns:
                out[col] = out[col].map(lambda x, s=spec: format(x, s))
    display(out)


## Пример работы

Порог поддержки 1%, достоверность 40%. Два способа сортировки.


In [ ]:
DEMO_SUPPORT = 0.01
DEMO_CONFIDENCE = 0.40

rules_by_support, itemsets_demo = mine_association_rules(
    transactions, DEMO_SUPPORT, DEMO_CONFIDENCE, order="support_desc"
)
rules_lex = association_rules(itemsets_demo, DEMO_CONFIDENCE, order="lex")

print(f"Частых наборов: {len(itemsets_demo)}")
print(f"Правил при conf ≥ {DEMO_CONFIDENCE:.0%}: {len(rules_by_support)}")

print("\nСортировка по убыванию поддержки:")
show_df(
    rules_to_frame(rules_by_support),
    {"поддержка": ".2%", "достоверность": ".2%"},
)

print("\nЛексикографическая сортировка (первые 15):")
show_df(
    rules_to_frame(rules_lex).head(15),
    {"поддержка": ".2%", "достоверность": ".2%"},
)


## Выбор порогов для экспериментов

В задании предложены поддержка **10%** и достоверность **70–95%**. На `baskets.csv` это даёт пустой список правил:

- при поддержке 10% и 15% есть только одиночные товары (нет частых пар) — из них правила не строятся;
- даже при поддержке 1% максимальная достоверность на этом наборе около **51%**, поэтому порог 70% отсекает все правила.

Поэтому:

1. для сверки с формулировкой задания прогоняем «как написано» (ожидаем нули);
2. основной эксперимент: **фиксируем поддержку 1%** (есть пары и тройки) и варьируем достоверность **от 20% до 50% с шагом 5%** — это тот же тип эксперимента, но в диапазоне, где правила реально появляются.


In [ ]:
print("Сверка с порогами из формулировки задания")
itemsets_10 = apriori(transactions, 0.10)
print(
    f"min_support=10%: частых наборов {len(itemsets_10)}, "
    f"из них длины ≥ 2: {sum(1 for s, _ in itemsets_10 if len(s) >= 2)}"
)

assign_rows = []
for conf in [i / 100 for i in range(70, 100, 5)]:
    t0 = perf_counter()
    found = association_rules(itemsets_10, conf)
    elapsed = perf_counter() - t0
    assign_rows.append(
        {
            "порог достоверности": f"{conf:.0%}",
            "число правил": len(found),
            "время генерации правил, с": elapsed,
        }
    )
show_df(pd.DataFrame(assign_rows), {"время генерации правил, с": ".6f"})


## Основной эксперимент

Фиксированная поддержка 1%. Порог достоверности: 20%, 25%, 30%, 35%, 40%, 45%, 50%.

Частые наборы считаются один раз; для каждого порога достоверности измеряется время **генерации правил** (именно этот шаг зависит от порога).


In [ ]:
FIXED_SUPPORT = 0.01
CONF_THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

t0 = perf_counter()
itemsets_fixed = apriori(transactions, FIXED_SUPPORT)
apriori_time = perf_counter() - t0
print(f"Apriori при min_support={FIXED_SUPPORT:.0%}: {len(itemsets_fixed)} наборов за {apriori_time:.3f} с")

exp_rows = []
for conf in CONF_THRESHOLDS:
    t0 = perf_counter()
    found = association_rules(itemsets_fixed, conf, order="support_desc")
    elapsed = perf_counter() - t0
    exp_rows.append(
        {
            "порог достоверности": f"{conf:.0%}",
            "min_confidence": conf,
            "число правил": len(found),
            "время, с": elapsed,
        }
    )

rules_exp_df = pd.DataFrame(exp_rows)
show_df(rules_exp_df.drop(columns=["min_confidence"]), {"время, с": ".6f"})


### Быстродействие поиска правил при изменяемом пороге достоверности


In [ ]:
fig, ax = plt.subplots()
ax.plot(
    rules_exp_df["min_confidence"] * 100,
    rules_exp_df["время, с"],
    marker="o",
    linewidth=2,
)
ax.set_title("Время генерации правил (поддержка 1%, baskets.csv)")
ax.set_xlabel("Порог достоверности, %")
ax.set_ylabel("Время, с")
ax.set_xticks(rules_exp_df["min_confidence"] * 100)
for x, y in zip(rules_exp_df["min_confidence"] * 100, rules_exp_df["время, с"]):
    ax.annotate(f"{y:.4f} с", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
fig.tight_layout()
plt.show()


### Число найденных правил при изменяемом пороге достоверности


In [ ]:
fig, ax = plt.subplots()
ax.plot(
    rules_exp_df["min_confidence"] * 100,
    rules_exp_df["число правил"],
    marker="o",
    linewidth=2,
)
ax.set_title("Число ассоциативных правил (поддержка 1%, baskets.csv)")
ax.set_xlabel("Порог достоверности, %")
ax.set_ylabel("Количество правил")
ax.set_xticks(rules_exp_df["min_confidence"] * 100)
for x, y in zip(rules_exp_df["min_confidence"] * 100, rules_exp_df["число правил"]):
    ax.annotate(str(int(y)), (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
fig.tight_layout()
plt.show()


## Правила с |антецедент| + |консеквент| ≤ 7

На выбранных порогах максимальная длина частого набора равна 3, поэтому **все** найденные правила удовлетворяют ограничению. Ниже — компактный список с достоверностью не ниже 40% (разумный объём для разбора).


In [ ]:
MAX_ITEMS = 7
compact = [
    r
    for r in association_rules(itemsets_fixed, 0.40, order="support_desc")
    if len(r["antecedent"]) + len(r["consequent"]) <= MAX_ITEMS
]
print(f"Правил с длиной A∪B ≤ {MAX_ITEMS} и conf ≥ 40%: {len(compact)}")
show_df(
    rules_to_frame(compact).drop(columns=["антецедент", "консеквент"]),
    {"поддержка": ".2%", "достоверность": ".2%"},
)


## Содержательный смысл

Почти все сильные правила ведут к **минеральной воде** или **макаронам** — это самые массовые товары в чеках (вода встречается примерно в 24% транзакций). Поэтому:

- **говяжий фарш, яйца → минеральная вода** (поддержка ~1.0%, достоверность ~50.7%): если в корзине уже фарш и яйца (заготовка на второе), в половине таких чеков есть ещё вода. Это не «вода покупается из‑за фарша», а типичный набор «продукты на ужин + напиток».
- **говяжий фарш, молоко → минеральная вода** (~1.1%, ~50.3%) и **говяжий фарш, шоколад → минеральная вода** (~1.1%, ~47.4%) — та же картина: фарш часто идёт в относительно крупных корзинах, где почти наверняка есть самый популярный товар.
- **суп → минеральная вода** (~2.3%, ~45.6%) и **оливковое масло → минеральная вода** (~2.8%, ~41.9%): готовые/базовые продукты тоже тянутся к воде.
- Обратные правила слабее: вода слишком частая, поэтому $\mathrm{conf}(\text{вода} \rightarrow X)$ невелика. Это ожидаемо: достоверность нормируется на поддержку антецедента.
- Пара **говяжий фарш → макароны** (~4.0%, ~41%) ближе к «рецепту»: фарш и макароны — сочетаемые ингредиенты блюда, а не только «популярный товар в большой корзине».
- **перец → макароны** (~1.0%, ~39%) похоже на приправу к гарниру.

Практический вывод для магазина: выкладка воды у кассы и рядом с мясом/макаронами согласуется с чеками; промо на фарш имеет смысл стыковать с макаронами. Высокой достоверности (70%+) на этом супермаркетном наборе нет: покупки разнообразные, жёстких «почти всегда вместе» связок почти не видно.

По графикам: при росте порога достоверности число правил монотонно падает (163 при 20% → 2 при 50%). Время генерации правил почти не меняется: отбор — это проход по уже посчитанным наборам, основная работа остаётся за Apriori.
